# Titanic Dataset Analysis
## AI Assignment 2 – Feature Engineering & Selection
**Objective:** Build a clean, feature-rich dataset ready for survival prediction modelling.

---


## 0. Setup & Imports

In [ ]:
import sys, os
sys.path.append('..')  # so we can import from scripts/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# nice plots
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

df_raw = pd.read_csv('../data/train.csv')
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()


---
## Part 1: Data Cleaning

### 1.1 – Inspect Missing Values
First thing I always do is get a full picture of what's missing before touching anything.


In [ ]:
missing = df_raw.isnull().sum()
pct     = (missing / len(df_raw) * 100).round(2)
missing_report = pd.DataFrame({"Missing Count": missing, "% Missing": pct})
missing_report[missing_report["Missing Count"] > 0]


### 1.2 – Missing Value Handling Strategy

| Column | % Missing | Strategy | Reason |
|--------|-----------|----------|--------|
| **Age** | ~20% | Median impute grouped by Pclass + Sex | Median is robust to skew; grouping gives better estimates than a global median |
| **Fare** | <1% | Global median impute | Only 1 row – any imputation method works fine here |
| **Embarked** | <1% | Mode impute | 2–3 rows, Southampton is overwhelmingly dominant |
| **Cabin** | ~77% | Extract deck letter + binary indicator, drop raw column | Too many missing to impute meaningfully; the deck letter still carries useful class info |


In [ ]:
df = df_raw.copy()

# Age – group median is more accurate than a flat overall median
age_median = df.groupby(["Pclass","Sex"])["Age"].transform("median")
df["Age"] = df["Age"].fillna(age_median)
df["Age"] = df["Age"].fillna(df["Age"].median())   # safety fallback

# Fare – just 1 row
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# Embarked – 2-3 rows
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Cabin – extract deck, add indicator, drop raw
df["HasCabin"] = df["Cabin"].notnull().astype(int)
df["Deck"]     = df["Cabin"].str[0].fillna("Unknown")
df.drop(columns=["Cabin"], inplace=True)

print("Missing values remaining:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nAll clear!" if df.isnull().sum().sum() == 0 else "Still some missing!")


### 1.3 – Outlier Detection & Handling

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["Fare"], bins=50, color="#4C72B0", edgecolor="white")
axes[0].set_title("Fare Distribution (before capping)")
axes[0].set_xlabel("Fare")

axes[1].hist(df["Age"], bins=40, color="#DD8452", edgecolor="white")
axes[1].set_title("Age Distribution")
axes[1].set_xlabel("Age")

plt.tight_layout()
plt.savefig("plots_fare_age_dist.png", dpi=150)
plt.show()

fare_99 = df["Fare"].quantile(0.99)
print(f"Fare 99th percentile: {fare_99:.2f}")
print(f"Fare max before cap:  {df['Fare'].max():.2f}")
df["Fare"] = df["Fare"].clip(upper=fare_99)
print(f"Fare max after cap:   {df['Fare'].max():.2f}")


**Decision:** Fare has a heavy right tail – a handful of luxury first-class tickets are well above the 99th percentile. 
These are genuine data points, not errors, but capping them prevents them from dominating distance-based models. 
Age looks fine (1–80 range, roughly gamma-distributed), no action needed there.


### 1.4 – Data Consistency Checks

In [ ]:
# Standardize Sex column
df["Sex"] = df["Sex"].str.lower().str.strip()
print("Sex values:", df["Sex"].unique())

# Check for duplicates
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
df.drop_duplicates(inplace=True)

# Pclass as int
df["Pclass"] = df["Pclass"].astype(int)

# Save cleaned dataset
df.to_csv('../data/train_cleaned.csv', index=False)
print(f"\nCleaned dataset saved → ../data/train_cleaned.csv")
print(f"Shape: {df.shape}")
df.head()


---
## Part 2: Feature Engineering

### 2.1 – Family Features


In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"]    = (df["FamilySize"] == 1).astype(int)

# visualize survival by family size
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

survival_by_family = df.groupby("FamilySize")["Survived"].mean()
survival_by_family.plot(kind="bar", ax=axes[0], color="#4C72B0", edgecolor="white")
axes[0].set_title("Survival Rate by Family Size")
axes[0].set_xlabel("FamilySize")
axes[0].set_ylabel("Survival Rate")
axes[0].axhline(df["Survived"].mean(), color="red", linestyle="--", label="Overall avg")
axes[0].legend()

df.groupby("IsAlone")["Survived"].mean().plot(
    kind="bar", ax=axes[1], color=["#4C72B0","#DD8452"], edgecolor="white"
)
axes[1].set_title("Survival Rate: Alone vs With Family")
axes[1].set_xticklabels(["With Family","Alone"], rotation=0)
axes[1].set_ylabel("Survival Rate")

plt.tight_layout()
plt.savefig("plots_family_survival.png", dpi=150)
plt.show()
print(df[["FamilySize","IsAlone"]].head(10))


**Observation:** Solo travellers and very large families (7+) had the lowest survival rates. 
The sweet spot seems to be families of 2–4. `IsAlone` cleanly captures the solo-traveller disadvantage.


### 2.2 – Title Extraction from Name

In [ ]:
df["Title"] = df["Name"].str.extract(r',\s*([^\.]+)\.', expand=False)

# also try fallback pattern
mask = df["Title"].isnull()
if mask.any():
    df.loc[mask, "Title"] = df.loc[mask, "Name"].str.extract(
        r'\b(Mr|Mrs|Miss|Master|Dr|Rev|Col|Capt|Lady|Major)\b', expand=False
    )

# normalize rare/variant titles
title_map = {
    "Mlle":"Miss","Ms":"Miss","Mme":"Mrs","Lady":"Rare","Countess":"Rare",
    "Capt":"Rare","Col":"Rare","Don":"Rare","Dr":"Rare",
    "Major":"Rare","Rev":"Rare","Sir":"Rare","Jonkheer":"Rare","Dona":"Rare"
}
df["Title"] = df["Title"].replace(title_map)

main_titles = {"Mr","Mrs","Miss","Master"}
df["Title"] = df["Title"].apply(lambda t: t if pd.notna(t) and t in main_titles else "Rare")

# survival by title
title_survival = df.groupby("Title")["Survived"].mean().sort_values(ascending=False)
title_survival.plot(kind="bar", color="#4C72B0", edgecolor="white", figsize=(8,4))
plt.title("Survival Rate by Title")
plt.ylabel("Survival Rate")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots_title_survival.png", dpi=150)
plt.show()

print(df["Title"].value_counts())


### 2.3 – Deck Extraction from Cabin
(Already done in cleaning step above – Deck column exists)

In [ ]:
print("Deck distribution:")
print(df["Deck"].value_counts())

deck_survival = df.groupby("Deck")["Survived"].mean().sort_values(ascending=False)
deck_survival.plot(kind="bar", color="#55A868", edgecolor="white", figsize=(10,4))
plt.title("Survival Rate by Deck")
plt.ylabel("Survival Rate")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots_deck_survival.png", dpi=150)
plt.show()


### 2.4 – Age Groups

In [ ]:
bins   = [0, 12, 17, 60, 100]
labels = ["Child","Teen","Adult","Senior"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels)

age_survival = df.groupby("AgeGroup", observed=True)["Survived"].mean()
age_survival.plot(kind="bar", color="#C44E52", edgecolor="white", figsize=(8,4))
plt.title("Survival Rate by Age Group")
plt.ylabel("Survival Rate")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots_agegroup_survival.png", dpi=150)
plt.show()
print(df["AgeGroup"].value_counts())


### 2.5 – Fare Per Person

In [ ]:
# Raw fare for grouped tickets is total for the whole group - per person is more accurate
df["FarePerPerson"] = df["Fare"] / df["FamilySize"].replace(0, 1)
print(df[["Fare","FamilySize","FarePerPerson"]].head(10))


### 2.6 – Log Transformations for Skewed Features

In [ ]:
df["Fare_log"]          = np.log1p(df["Fare"])
df["FarePerPerson_log"] = np.log1p(df["FarePerPerson"])
df["Age_log"]           = np.log1p(df["Age"])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

df["Fare"].hist(ax=axes[0,0], bins=50, color="#4C72B0", edgecolor="white")
axes[0,0].set_title("Fare (original)")

df["Fare_log"].hist(ax=axes[0,1], bins=50, color="#4C72B0", edgecolor="white")
axes[0,1].set_title("Fare (log1p) – much better!")

df["Age"].hist(ax=axes[1,0], bins=40, color="#DD8452", edgecolor="white")
axes[1,0].set_title("Age (original)")

df["Age_log"].hist(ax=axes[1,1], bins=40, color="#DD8452", edgecolor="white")
axes[1,1].set_title("Age (log1p)")

plt.tight_layout()
plt.savefig("plots_log_transforms.png", dpi=150)
plt.show()


### 2.7 – Categorical Encoding (One-Hot)

In [ ]:
ohe_cols = ["Sex","Embarked","Title","Deck","AgeGroup"]
df_encoded = pd.get_dummies(df, columns=ohe_cols, drop_first=False)

# bool → int
bool_cols = df_encoded.select_dtypes(include="bool").columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"Shape before OHE: {df.shape}")
print(f"Shape after  OHE: {df_encoded.shape}")


### 2.8 – Interaction Features

In [ ]:
# Pclass x Fare: a high fare in third class is unusual and might signal something.
# Combining the two captures that nuance.
df_encoded["Pclass_x_Fare"] = df_encoded["Pclass"] * df_encoded["Fare_log"]

# drop columns we don't need anymore
drop_cols = ["Name","Ticket","PassengerId"]
df_encoded.drop(columns=[c for c in drop_cols if c in df_encoded.columns], inplace=True)

df_encoded.to_csv('../data/train_engineered.csv', index=False)
print(f"Engineered dataset saved → ../data/train_engineered.csv")
print(f"Shape: {df_encoded.shape}")
list(df_encoded.columns)


---
## Part 3: Feature Selection

### 3.1 – Correlation Analysis


In [ ]:
num_df = df_encoded.select_dtypes(include=[np.number])
corr   = num_df.corr().abs()

plt.figure(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, linewidths=0.3,
            annot=False, square=False)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("plots_correlation_matrix.png", dpi=150)
plt.show()

# identify highly correlated pairs (> 0.90)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_pairs = [(col, row) for col in upper.columns for row in upper.index
                   if upper.loc[row, col] > 0.90]
print("High correlation pairs (>0.90):", high_corr_pairs)


In [ ]:
# drop one from each high-corr pair
to_drop_corr = list({pair[0] for pair in high_corr_pairs})
print("Dropping:", to_drop_corr)
df_selected = df_encoded.drop(columns=to_drop_corr, errors="ignore")
print(f"Shape after dropping correlated features: {df_selected.shape}")


### 3.2 – Random Forest Feature Importance

In [ ]:
X = df_selected.drop(columns=["Survived"])
y = df_selected["Survived"]

# handle any leftover object columns
for col in X.select_dtypes(include="object").columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

importance_df = pd.DataFrame({
    "feature":    X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

# plot top 20
top20 = importance_df.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20["feature"][::-1], top20["importance"][::-1], color="#4C72B0")
ax.set_xlabel("Feature Importance")
ax.set_title("Random Forest Feature Importance (Top 20)")
plt.tight_layout()
plt.savefig("plots_feature_importance.png", dpi=150)
plt.show()

importance_df.head(20)


### 3.3 – Final Feature Selection & Justification

In [ ]:
# keep features with importance >= 1% OR in must-keep set
must_keep = {"Pclass","Fare_log","Age_log","FamilySize","IsAlone","Survived"}
threshold = 0.01

selected_feats = set(importance_df[importance_df["importance"] >= threshold]["feature"].tolist())
selected_feats = selected_feats | (must_keep & set(df_selected.columns))
selected_feats.add("Survived")

final_cols = [c for c in df_selected.columns if c in selected_feats]
df_final   = df_selected[final_cols]

df_final.to_csv('../data/train_selected.csv', index=False)
print(f"Final selected dataset: {df_final.shape}")
print("\nSelected features:")
for f in sorted(final_cols):
    imp = importance_df[importance_df['feature']==f]['importance'].values
    imp_str = f"{imp[0]:.4f}" if len(imp)>0 else "target"
    print(f"  {f:<35} {imp_str}")


### Feature Justification Summary

| Feature | Reason Kept |
|---------|-------------|
| `Pclass` | Strong proxy for socioeconomic status; well-known survival predictor |
| `Fare_log` | Direct measure of wealth/class; log transform normalizes skew |
| `Age_log` | Children had priority in lifeboats; clear non-linear effect |
| `FamilySize` | Solo travellers and huge families had lower survival |
| `IsAlone` | Captures the solo-traveller penalty cleanly |
| `Title_*` | Encodes gender + social class + approximate age simultaneously |
| `Sex_*` | Women and children first – strongest single predictor |
| `Pclass_x_Fare` | Interaction: price relative to class peers is more informative than raw price |
| `HasCabin` | Proxy for wealth (known cabin = bought a cabin = richer) |
| `Deck_*` | Proximity to lifeboats; higher decks had faster evacuation access |

**Features dropped:**
- Raw `Fare`, `Age`, `FarePerPerson` – kept log-transformed versions instead
- High-correlation pairs – redundant information
- `SibSp`, `Parch` – captured by the more informative `FamilySize`
- `Embarked_*` – low importance from RF, weak predictor after other features are included


---
## Summary

The pipeline goes: **raw data → clean → engineer features → select best subset**.

Key decisions:
1. Grouped median imputation for Age (better than global median)
2. Deck extraction from Cabin before dropping it (preserves signal)  
3. Log transforms on Fare and FarePerPerson (essential for linear models)
4. Title extraction – arguably the most engineered feature, encodes 3 variables in 1
5. RF importance + correlation filtering to trim to a lean, non-redundant feature set

The final `train_selected.csv` is ready to go into a classifier.
